# Phase 4 v0.7 Round — Kaggle CUDA

**v0.7 변경**: calibration weight `0.1 → 0.3` (coverage 개선 목적)  
모델 구조·데이터·optimizer는 v0.6과 동일 (Mode1Head in_dim=384, I_obs 1ch)

**실행 전 체크리스트**
- 우측 **Data** 탭 → **Add data** → `donghyun51/lens-phase4-v0-4` 추가
- Accelerator: **GPU T4 x1 이상**
- Internet: **On**

## Cell 1 — GPU 확인

In [ ]:
import subprocess, torch
print(subprocess.check_output(['nvidia-smi'], text=True))
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## Cell 2 — Input 경로 진단

`/kaggle/input/` 아래 실제 구조를 확인하고 파일을 찾는다.

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')

print('=== /kaggle/input/ 구조 ===')
if INPUT_ROOT.exists():
    for d in sorted(INPUT_ROOT.iterdir()):
        files = list(d.glob('*'))
        print(f'  {d.name}/')
        for f in sorted(files)[:6]:
            size_mb = f.stat().st_size / 1e6 if f.is_file() else 0
            print(f'    {f.name}  ({size_mb:.1f} MB)' if f.is_file() else f'    {f.name}/')
else:
    print('  /kaggle/input 없음 — 로컬 실행 중?')

## Cell 3 — 파일 경로 자동 설정

Cell 2 출력에서 마운트된 폴더명을 확인한 뒤,  
아래 `DATASET_SLUG`를 실제 폴더명으로 수정한다 (기본값: `lens-phase4-v0-4`).

In [ ]:
from pathlib import Path

DATASET_SLUG = 'lens-phase4-v0-4'   # ← Cell 2 출력에서 확인한 폴더명

INPUT_DIR = Path('/kaggle/input') / DATASET_SLUG
WORK_DIR  = Path('/kaggle/working')

def find_file(name: str) -> Path | None:
    p = INPUT_DIR / name
    if p.exists(): return p
    matches = list(Path('/kaggle/input').rglob(name))
    return matches[0] if matches else None

TRAIN_H5   = find_file('phase4_v0_4.h5')
UNFILT_H5  = find_file('phase4_v0_4_eval_unfiltered.h5')
SCALER_PKL = find_file('target_scaler_phase4_v0_4.pkl')

print('TRAIN_H5  :', TRAIN_H5)
print('UNFILT_H5 :', UNFILT_H5)
print('SCALER_PKL:', SCALER_PKL)

assert TRAIN_H5,   'phase4_v0_4.h5 를 찾을 수 없음 — Data 탭에서 donghyun51/lens-phase4-v0-4 추가 확인'
assert UNFILT_H5,  'phase4_v0_4_eval_unfiltered.h5 를 찾을 수 없음'
assert SCALER_PKL, 'target_scaler_phase4_v0_4.pkl 를 찾을 수 없음'

print('\nAll input files found OK ✅')
for p in [TRAIN_H5, UNFILT_H5, SCALER_PKL]:
    print(f'  {p.name}: {p.stat().st_size/1e6:.1f} MB')

## Cell 4 — 코드 클론 (GitHub main)

In [ ]:
import subprocess
from pathlib import Path

REPO_URL  = 'https://github.com/dasbaq/GV.git'   # ← 실제 repo URL로 수정
REPO_ROOT = Path('/kaggle/working/repo')
PROJ_DIR  = REPO_ROOT / 'Gravitational_Lens_MultiMode'

if not REPO_ROOT.exists():
    print('Cloning...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)], check=True)
else:
    print('Repo exists, pulling...')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

# v0.7 핵심 파일 확인
checks = {
    'v0.7 round script':  PROJ_DIR / 'scripts/phase4_v0_7_round.py',
    'round_eval':         PROJ_DIR / 'ml/training/round_eval.py',
    'physics_pairing':    PROJ_DIR / 'ml/training/physics_pairing.py',
    'encoders.py':        PROJ_DIR / 'ml/models/encoders.py',
    'error_corrector.py': PROJ_DIR / 'ml/models/error_corrector.py',
}
for label, path in checks.items():
    print(f'  {"✅" if path.exists() else "❌"} {label}')

# v0.7 calibration weight 확인
round_src = (PROJ_DIR / 'scripts/phase4_v0_7_round.py').read_text()
if '_CALIBRATION_WEIGHT_V07 = 0.3' in round_src:
    print('  ✅ calibration weight = 0.3 확인')
else:
    print('  ❌ calibration weight 확인 실패 — push 확인 필요')

# ImageEncoder 확인
enc_src = (PROJ_DIR / 'ml/models/encoders.py').read_text()
print(f'  {"✅" if "ImageEncoder" in enc_src else "❌"} ImageEncoder (I_obs 1ch)')

# Mode3 삭제 확인
gone = not (PROJ_DIR / 'inversion/mode3_wrapper.py').exists()
print(f'  {"✅" if gone else "❌"} mode3_wrapper GONE')

## Cell 5 — 환경변수 + import 검증

In [ ]:
import os, sys, subprocess

os.environ['LENS_DATA_PATH']            = str(TRAIN_H5)
os.environ['LENS_DATA_PATH_UNFILTERED'] = str(UNFILT_H5)
os.environ['LENS_SCALER_PATH']          = str(SCALER_PKL)
os.environ['LENS_WORK_ROOT']            = str(WORK_DIR)

os.chdir(str(PROJ_DIR))
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))

print('cwd:', os.getcwd())
for k in ['LENS_DATA_PATH','LENS_DATA_PATH_UNFILTERED','LENS_SCALER_PATH','LENS_WORK_ROOT']:
    print(f'  {k} = {os.environ[k]}')

subprocess.run(['pip', 'install', '-q', 'h5py', 'scipy', 'pyyaml'], check=True)

from ml.models.error_corrector import MultiModalErrorCorrector
from ml.models.encoders import ImageEncoder
from ml.training.round_eval import evaluate_mode1_h0_on_loader
from ml.training.physics_pairing import add_paired_physics_predictions
print('All imports OK ✅')

# v0.7 calibration weight 설정 확인 (load_cfg 호출)
import yaml
sys.path.insert(0, str(PROJ_DIR))
import importlib.util, types
spec = importlib.util.spec_from_file_location(
    'phase4_v0_7_round',
    str(PROJ_DIR / 'scripts/phase4_v0_7_round.py')
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
cfg = mod.load_cfg()
cal_w = cfg['training']['loss_weights']['calibration']
print(f'calibration weight: {cal_w}  (기대: 0.3)')
assert cal_w == 0.3, f'calibration weight mismatch: {cal_w}'

# v0.7 모델 구조 빠른 확인
mc = dict(cfg['model'])
mc['param_in_dim'] = len(cfg['data']['param_normalization']) + 5
m = MultiModalErrorCorrector(mc)
w = m.state_dict()['head1.net.0.weight']
print(f'head1.net.0.weight: {w.shape}  (기대: [64, 384])')
assert w.shape == (64, 384), f'in_dim mismatch: {w.shape}'
print('v0.7 architecture & calibration weight verified ✅')

## Cell 6 — 2-epoch sanity run

NaN 없이 완료되면 Full run 진행.  
`nan_detected=false` 확인 후 Cell 7 실행.

In [ ]:
import os
from pathlib import Path

PROJ_DIR     = Path('/kaggle/working/repo/Gravitational_Lens_MultiMode')
ROUND_SCRIPT = str(PROJ_DIR / 'scripts' / 'phase4_v0_7_round.py')

if not Path(ROUND_SCRIPT).exists():
    raise FileNotFoundError(f'Script not found: {ROUND_SCRIPT}  -- run Cell 4 first')

os.environ.setdefault('LENS_WORK_ROOT', '/kaggle/working')
print('ROUND_SCRIPT:', ROUND_SCRIPT)

!python {ROUND_SCRIPT} --phase train --device cuda --workers 0 --epochs 2 --bootstrap-n 0

## Cell 7 — Full run (50 epochs)

sanity에서 `nan_detected=false` 확인 후 실행.  
예상 시간: ~40-80분 (T4 기준).

In [ ]:
import os
from pathlib import Path

PROJ_DIR     = Path('/kaggle/working/repo/Gravitational_Lens_MultiMode')
ROUND_SCRIPT = str(PROJ_DIR / 'scripts' / 'phase4_v0_7_round.py')

if not Path(ROUND_SCRIPT).exists():
    raise FileNotFoundError(f'Script not found: {ROUND_SCRIPT}  -- run Cell 4 first')

os.environ.setdefault('LENS_WORK_ROOT', '/kaggle/working')
print('ROUND_SCRIPT:', ROUND_SCRIPT)

!python {ROUND_SCRIPT} --phase train --device cuda --workers 0 --epochs 50 --bootstrap-n 1000

## Cell 8 — 결과 확인 (acceptance report)

**주목 지표**: coverage (목표 [0.62, 0.78])  
v0.6 대비 비교: RMSE 5.258 / r 0.503 / coverage **0.52 → 개선 여부 확인**

In [ ]:
import json
from pathlib import Path
WORK_DIR = Path('/kaggle/working')
logs_dir = WORK_DIR / 'logs'

for fname in ['phase4_v0_7_imgres_h0_eval.json',
              'phase4_v0_7_imgres_h0_eval_unfiltered.json',
              'phase4_v0_7_infra_equivalence.json']:
    p = logs_dir / fname
    if not p.exists():
        print(f'⏳ {fname}: not found yet')
        continue
    data = json.loads(p.read_text())
    print(f'\n== {fname} ==')
    if 'stage_b_acceptance_report' in data:
        rep = data['stage_b_acceptance_report']
        print('all_pass:', rep.get('all_pass_excluding_record_only'),
              '| leak_triggered:', rep.get('leak_triggered'))
        for row in rep.get('pass_rows', []):
            mark = '✅' if row['pass'] else ('📝' if row['pass'] is None else '❌')
            print(f"  {mark} {row['metric']}: {row['value']}")
    elif 'best' in data:
        m   = data['best']['mode1']['h0']['model']
        cal = data['best']['mode1']['log_sigma_calibration']
        cov = cal['coverage_abs_residual_le_1sigma']
        cov_ci = cal['coverage_abs_residual_le_1sigma_ci95_clopper_pearson']
        print(f"  RMSE={m['RMSE']:.3f}  r={m['r']:.4f}  bias={m['bias']:.3f}")
        print(f"  coverage={cov:.3f}  CI={cov_ci}")
        mark = '✅' if 0.62 <= cov <= 0.78 else '❌'
        print(f"  {mark} coverage target [0.62, 0.78]")

print('\n== v0.6 vs v0.7 비교 목표 ==')
print('  v0.6: RMSE=5.258  r=0.503  coverage=0.52  ← 개선 대상')
print('  v0.7: coverage [0.62, 0.78] 진입 목표 (RMSE/r 소폭 하락 허용)')

## Cell 9 — Checkpoint v0.7 검증

- `head1.net.0.weight` shape `(64, 384)` = v0.7(=v0.6) OK  
- `par_enc.net.0.weight` shape `(256, 20)` = 20-dim ParamEncoder OK  
- `img_enc.*` 키 있음 = I_obs 1ch 복구 확인  
- `head3.*` 키 없음 = Mode 3 삭제 유지 확인

In [ ]:
import torch
from pathlib import Path
WORK_DIR = Path('/kaggle/working')

ckpt_path = WORK_DIR / 'checkpoints' / 'phase4_v0_7_imgres_best.pt'

if not ckpt_path.exists():
    print(f'checkpoint not found: {ckpt_path}')
    print('Full run (Cell 7) 완료 후 다시 실행하세요.')
else:
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    w        = sd.get('head1.net.0.weight')
    par      = sd.get('par_enc.net.0.weight')
    img_enc1 = sd.get('img_enc.enc1.net.0.weight')
    img_keys = [k for k in sd if k.startswith('img_enc.')]
    h3_keys  = [k for k in sd if k.startswith('head3.')]

    print(f'head1.net.0.weight         : {w.shape}')        # 기대: (64, 384)
    print(f'par_enc.net.0.weight       : {par.shape}')       # 기대: (256, 20)
    if img_enc1 is not None:
        print(f'img_enc.enc1.net.0.weight  : {img_enc1.shape}')  # 기대: (32, 1, 3, 3)
    print(f'img_enc keys (기대 >0)     : {len(img_keys)}')
    print(f'head3  keys  (기대 0)      : {len(h3_keys)}')

    assert w.shape == (64, 384),   f'Expected (64,384), got {w.shape}'
    assert par.shape[1] == 20,     f'Expected par_enc 20-dim, got {par.shape}'
    assert len(img_keys) > 0,      'img_enc keys must be present'
    assert len(h3_keys)  == 0,     f'head3 keys found: {h3_keys[:3]}'
    print('\nv0.7 checkpoint verified ✅')

## Cell 10 — 산출물 복사 및 목록 (회수 대상)

Output 탭에서 직접 다운로드하거나 `fetch_kaggle_results.py`로 회수.

In [ ]:
import shutil
from pathlib import Path
WORK_DIR = Path('/kaggle/working')

artifacts = [
    WORK_DIR / 'checkpoints' / 'phase4_v0_7_imgres_best.pt',
    WORK_DIR / 'logs' / 'phase4_v0_7_imgres_h0_eval.json',
    WORK_DIR / 'logs' / 'phase4_v0_7_imgres_h0_eval_unfiltered.json',
    WORK_DIR / 'logs' / 'phase4_v0_7_imgres_long_history.json',
    WORK_DIR / 'logs' / 'phase4_v0_7_infra_equivalence.json',
]

print('== 산출물 존재 여부 ==')
for p in artifacts:
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else '아직 없음'
    print(f'  {"✅" if p.exists() else "⏳"} {p.name}  ({size})')

# Output 탭에서 다운로드 가능하도록 루트로 복사
print('\n== /kaggle/working/ 루트로 복사 (Output 탭 다운로드용) ==')
for p in artifacts:
    if p.exists():
        dst = WORK_DIR / p.name
        shutil.copy2(p, dst)
        print(f'  copied → {dst.name}')
    else:
        print(f'  skip (없음): {p.name}')

print('\n== 로컬 저장 위치 ==')
print('  phase4_v0_7_imgres_best.pt     → data/checkpoints/')
print('  phase4_v0_7_imgres_*.json      → data/logs/')